In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# ⭐ Customer Review Analyzer

## What We're Building

A tool that analyzes a collection of customer reviews and produces:
1. **Sentiment Analysis** — Positive, negative, neutral for each review
2. **Theme Extraction** — Group reviews by common topics (quality, shipping, price...)
3. **Top Complaints** — Most frequent negative themes
4. **Executive Summary** — Overall customer satisfaction report

## How It Works

```
Reviews → Embed each review → Cluster by similarity → Label clusters
    ↓
Analyze sentiment per review → Aggregate by theme → Generate report
```

## Key Concepts
- **Embeddings for clustering**: Group similar reviews by meaning
- **Batch processing**: Analyze many reviews efficiently
- **Structured analysis**: Extract consistent data from unstructured text

## Stack
- **LangChain** — LLM chains for sentiment and theme analysis
- **Ollama** — local LLM and embeddings
- **ChromaDB** — vector store for similarity-based grouping

## Step 1 — Install & Import Dependencies

In [ ]:
# Install if needed (uncomment)
# !pip install langchain langchain-ollama langchain-community chromadb -q

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from pydantic import BaseModel, Field
import json
from collections import Counter

print("All imports successful!")

## Step 2 — Sample Review Dataset

Here's a realistic set of product reviews for a wireless headphone.

In [ ]:
reviews = [
    {
        "id": 1,
        "rating": 5,
        "text": "Amazing sound quality! The noise cancellation is top-notch. "
                "I use these for my daily commute and they block out everything. "
                "Battery lasts all week with moderate use. Best headphones I've owned.",
    },
    {
        "id": 2,
        "rating": 2,
        "text": "The sound is decent but the Bluetooth keeps disconnecting from my phone. "
                "Happens every 20-30 minutes. Very frustrating during calls. "
                "Customer support was unhelpful.",
    },
    {
        "id": 3,
        "rating": 4,
        "text": "Great headphones overall. Comfortable for long wearing sessions. "
                "The app could use some work though - it's clunky and slow. "
                "Sound quality and build are excellent.",
    },
    {
        "id": 4,
        "rating": 1,
        "text": "Returned after 3 days. The headband broke while adjusting the size. "
                "Cheap plastic. For the price point, I expected much better build quality. "
                "Waste of money.",
    },
    {
        "id": 5,
        "rating": 5,
        "text": "Perfect for working from home! Crystal clear mic quality for Zoom calls. "
                "My colleagues say I sound way better than with my old headset. "
                "Comfortable enough to wear 8 hours straight.",
    },
    {
        "id": 6,
        "rating": 3,
        "text": "Average headphones. Nothing special about the sound for the price. "
                "Battery life is good (about 30 hours). The case is nice. "
                "I expected more given the hype.",
    },
    {
        "id": 7,
        "rating": 1,
        "text": "Shipping was a disaster. Arrived 2 weeks late and the box was crushed. "
                "Product was damaged. Tried to get a refund but the process is incredibly "
                "slow. Still waiting after 10 days.",
    },
    {
        "id": 8,
        "rating": 4,
        "text": "Sound quality rivals headphones twice the price. ANC is effective on "
                "planes. Only complaint is they get warm after a couple hours. "
                "Would buy again.",
    },
    {
        "id": 9,
        "rating": 2,
        "text": "The EQ settings in the app don't save properly. Every time I restart "
                "the app, I have to reconfigure everything. Also the firmware update "
                "bricked my headphones for an hour. Software team needs to do better.",
    },
    {
        "id": 10,
        "rating": 5,
        "text": "Got these as a gift and they exceeded expectations. Sleek design, "
                "premium feel, amazing bass response. Even my audiophile friend was impressed. "
                "The transparency mode is genuinely useful.",
    },
]

print(f"Loaded {len(reviews)} reviews")
print(f"Rating distribution: {Counter(r['rating'] for r in reviews)}")

## Step 3 — Individual Review Analysis

Analyze each review for sentiment, themes, and key phrases.

In [ ]:
class ReviewAnalysis(BaseModel):
    """Analysis of a single customer review."""
    sentiment: str = Field(description="positive, negative, neutral, or mixed")
    sentiment_score: float = Field(description="Score from -1.0 (very negative) to 1.0 (very positive)")
    themes: list[str] = Field(description="Main topics: sound_quality, build_quality, comfort, "
                              "battery, bluetooth, app, shipping, customer_service, price_value, design")
    key_phrases: list[str] = Field(description="Important phrases from the review")
    is_actionable: bool = Field(description="Whether this review contains actionable feedback")


# Initialize
llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)  # Low temp for consistent analysis
review_parser = JsonOutputParser(pydantic_object=ReviewAnalysis)

review_prompt = ChatPromptTemplate.from_template(
    """Analyze this customer review for a wireless headphone product.

{format_instructions}

Review (Rating: {rating}/5):
"{review_text}"

Analysis as JSON:"""
)

review_chain = review_prompt | llm | review_parser

print("Review analysis chain ready!")

In [ ]:
# Analyze all reviews
analyses = []

for review in reviews:
    print(f"Analyzing review #{review['id']}... ", end="")
    analysis = review_chain.invoke({
        "rating": review["rating"],
        "review_text": review["text"],
        "format_instructions": review_parser.get_format_instructions(),
    })
    analysis["review_id"] = review["id"]
    analysis["rating"] = review["rating"]
    analyses.append(analysis)
    print(f"{analysis.get('sentiment', '?')} ({analysis.get('sentiment_score', 0):+.1f})")

print(f"\n✅ Analyzed all {len(analyses)} reviews")

## Step 4 — Aggregate Results

Now we aggregate individual analyses to find patterns.

In [ ]:
# Sentiment distribution
sentiments = Counter(a.get("sentiment", "unknown") for a in analyses)
print("📊 SENTIMENT DISTRIBUTION")
for sentiment, count in sentiments.most_common():
    bar = "█" * (count * 5)
    print(f"  {sentiment:10s} {bar} ({count})")

# Average sentiment score
avg_score = sum(a.get("sentiment_score", 0) for a in analyses) / len(analyses)
print(f"\n  Average score: {avg_score:+.2f}")

# Theme frequency
all_themes = []
for a in analyses:
    all_themes.extend(a.get("themes", []))

theme_counts = Counter(all_themes)
print(f"\n📋 THEME FREQUENCY")
for theme, count in theme_counts.most_common(10):
    bar = "█" * (count * 3)
    print(f"  {theme:20s} {bar} ({count})")

In [ ]:
# Find top complaints (negative reviews' themes)
negative_themes = []
for a in analyses:
    if a.get("sentiment") in ("negative", "mixed"):
        negative_themes.extend(a.get("themes", []))

complaint_counts = Counter(negative_themes)
print("🚨 TOP COMPLAINTS (themes from negative/mixed reviews)")
for theme, count in complaint_counts.most_common(5):
    print(f"  • {theme}: {count} mentions")

# Actionable feedback
actionable = [a for a in analyses if a.get("is_actionable")]
print(f"\n📌 ACTIONABLE REVIEWS: {len(actionable)}/{len(analyses)}")
for a in actionable:
    print(f"  Review #{a['review_id']}: {', '.join(a.get('key_phrases', [])[:2])}")

## Step 5 — Generate Executive Summary

Feed all analysis results to the LLM for a comprehensive summary.

In [ ]:
exec_prompt = ChatPromptTemplate.from_template(
    """You are a product analytics expert. Based on the customer review analysis data below,
write an executive summary report.

Include:
1. **Overall Sentiment** — How customers feel about the product
2. **Strengths** — What customers love most
3. **Weaknesses** — Top areas of complaint
4. **Recommendations** — 3-5 actionable improvements prioritized by impact
5. **Risk Assessment** — Any critical issues that need immediate attention

Analysis Data:
- Total reviews: {total_reviews}
- Average sentiment score: {avg_score}
- Sentiment distribution: {sentiment_dist}
- Top themes: {top_themes}
- Top complaints: {top_complaints}
- Actionable reviews: {actionable_count}

Individual review summaries:
{review_summaries}

Executive Summary:"""
)

exec_chain = exec_prompt | llm | StrOutputParser()

# Prepare review summaries
review_summaries = "\n".join(
    f"- Review #{a['review_id']} (Rating: {a['rating']}/5, {a.get('sentiment', '?')}): "
    f"Themes: {', '.join(a.get('themes', []))}. Key: {', '.join(a.get('key_phrases', [])[:2])}"
    for a in analyses
)

print("📝 Generating executive summary...\n")
exec_summary = exec_chain.invoke({
    "total_reviews": len(analyses),
    "avg_score": f"{avg_score:+.2f}",
    "sentiment_dist": dict(sentiments),
    "top_themes": dict(theme_counts.most_common(5)),
    "top_complaints": dict(complaint_counts.most_common(5)),
    "actionable_count": len(actionable),
    "review_summaries": review_summaries,
})

print(exec_summary)

## Step 6 — Semantic Grouping with Embeddings

Use embeddings to group reviews by semantic similarity,
discovering themes the LLM might miss.

In [ ]:
# Embed all reviews
embeddings_model = OllamaEmbeddings(model="nomic-embed-text-v2-moe")

review_texts = [r["text"] for r in reviews]
review_vectors = embeddings_model.embed_documents(review_texts)

print(f"Embedded {len(review_vectors)} reviews")
print(f"Vector dimension: {len(review_vectors[0])}")

# Calculate similarity between reviews
import numpy as np

vectors = np.array(review_vectors)
# Cosine similarity
norms = np.linalg.norm(vectors, axis=1, keepdims=True)
normalized = vectors / norms
similarity_matrix = normalized @ normalized.T

print(f"\n📊 MOST SIMILAR REVIEW PAIRS:")
pairs = []
for i in range(len(reviews)):
    for j in range(i + 1, len(reviews)):
        pairs.append((similarity_matrix[i, j], i, j))

pairs.sort(reverse=True)
for sim, i, j in pairs[:5]:
    print(f"  Reviews #{reviews[i]['id']} & #{reviews[j]['id']}: "
          f"similarity={sim:.3f}")
    print(f"    #{reviews[i]['id']}: \"{reviews[i]['text'][:60]}...\"")
    print(f"    #{reviews[j]['id']}: \"{reviews[j]['text'][:60]}...\"")
    print()

## Step 7 — Complete Analysis Pipeline

In [ ]:
def analyze_reviews(reviews: list[dict]) -> dict:
    """
    Complete review analysis pipeline.
    
    Args:
        reviews: List of {id, rating, text} dicts
    
    Returns:
        Dict with individual analyses, aggregates, and executive summary
    """
    print(f"\n{'='*60}")
    print(f"📊 CUSTOMER REVIEW ANALYSIS")
    print(f"   Analyzing {len(reviews)} reviews...")
    print(f"{'='*60}\n")
    
    # Analyze each review
    all_analyses = []
    for review in reviews:
        result = review_chain.invoke({
            "rating": review["rating"],
            "review_text": review["text"],
            "format_instructions": review_parser.get_format_instructions(),
        })
        result["review_id"] = review["id"]
        result["rating"] = review["rating"]
        all_analyses.append(result)
        s = result.get('sentiment', '?')
        print(f"  ✓ Review #{review['id']} → {s}")
    
    # Aggregate
    sentiment_dist = Counter(a.get("sentiment", "unknown") for a in all_analyses)
    all_themes = [t for a in all_analyses for t in a.get("themes", [])]
    theme_dist = Counter(all_themes)
    
    print(f"\n✅ Analysis complete!")
    print(f"   Sentiments: {dict(sentiment_dist)}")
    print(f"   Top themes: {dict(theme_dist.most_common(5))}")
    
    return {
        "analyses": all_analyses,
        "sentiment_distribution": dict(sentiment_dist),
        "theme_distribution": dict(theme_dist),
    }


# Usage (already analyzed above, so just showing the function)
print("Pipeline function ready!")
print("Usage: result = analyze_reviews(reviews)")

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **Batch Analysis** | Process many reviews through the same chain |
| **Structured Output** | Consistent JSON format for every review |
| **Aggregation** | Combine individual results into trends/patterns |
| **Embedding Similarity** | Find semantically similar reviews without LLM |
| **Multi-stage Pipeline** | Analyze → Aggregate → Summarize |
| **Low Temperature** | Ensures consistent, deterministic analysis |

## 🔧 Customization Ideas

- **Competitor comparison**: Analyze reviews for competing products
- **Trend over time**: Track sentiment changes across review dates
- **Response generator**: Auto-draft replies to negative reviews
- **Dashboard**: Visualize results with matplotlib/plotly
- **Scale**: Use LangChain's batch API for parallel processing